In [ ]:
#Initiating autoreload
%load_ext autoreload
%autoreload 2
%matplotlib inline

#Adding parent directory to working path
import os
import sys

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)

#Improting packages
from astropy.visualization import AsinhStretch, ImageNormalize, simple_norm
from complexLA.Analysis import paltas_model, metrics, network_predictions
import corner
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import multivariate_normal

### 2) Paltas Models

Next we will draw images of our lenses and visually inspect them.

In [ ]:
config_file='../complexLA/Configs/main_deflector_config.py'
im_wop, metadata = paltas_model.PaltasModel(config_file)
plt.axis('off')
plt.imshow(im_wop,norm=simple_norm(im_wop,stretch='log',min_cut=1e-6))

In [ ]:
config_file='../complexLA/Configs/dark_perturber_config.py'
im_wp, metadata = paltas_model.PaltasModel(config_file)
plt.axis('off')
plt.imshow(im_wp,norm=simple_norm(im_wp,stretch='log',min_cut=1e-6))

In [ ]:
config_file='../complexLA/Configs/luminous_perturber_config.py'
im_wpl, metadata = paltas_model.PaltasModel(config_file)
plt.axis('off')
plt.imshow(im_wpl,norm=simple_norm(im_wpl,stretch='log',min_cut=1e-6))

### 2.1) Residuals ###

We will plot the residual between the image with a luminous perturber added to the main deflector and the image with just a main deflector to check that the image positions of the source are changing.

Luminous Perturber Residual

In [ ]:
#Defining im_ris
im_ris_wpl = im_wpl-im_wop
resid_norm =mcolors.TwoSlopeNorm(vmin=-0.025,vcenter=0,vmax=0.025)

#Plotting residual
plt.axis('off')
plt.imshow(im_ris_wpl,norm=resid_norm,cmap='bwr')

### 3) Predict the Lens Parameters for Each Test Set ###

This module makes neural posterior estimation predictions of the ten lens parameters of interest. The output gives the truth values, or known values of the lens (y_test), the predicted values (y_pred_...), the standard deviation of the posterior (std_pred_...), and the predicted precision matrix (prec_pred_...).

The image paths used are the same folders that the images of the lenses were saved to in the Test Set Generation notebook. 

If you still need to generate test sets of lenses please run that notebook first.

In [ ]:
#Defining sample number in case the above parameter generation was not ran
sample_num = 101

#Parameter predictions for test set without a perturber
image_path_main_deflector = '../images_for_network/main_deflector/'
y_test, y_pred_wop, std_pred_wop, prec_pred_wop = network_predictions.Predictions(image_path_main_deflector)

#Parameter predictions for test set with a dark perturber
image_path_dark_perturber = '../images_for_network/dark_perturber/'
y_test, y_pred_wp, std_pred_wp, prec_pred_wp = network_predictions.Predictions(image_path_dark_perturber)

#Parameter predictions for test set with a luminous perturber
image_path_luminous_perturber = '../images_for_network/luminous_perturber/'
y_test, y_pred_wpl, std_pred_wpl, prec_pred_wpl = network_predictions.Predictions(image_path_luminous_perturber)

### 3.1) Residuals ###
Here we will look at the residuals between the test sets of interest and the test set of simple main deflectors.

In [ ]:
#Populating image arrays for use later
im_md = []
im_dp = []
im_lp = []

for file in os.listdir(image_path_main_deflector):
    if file.endswith('.npy'):
        im_md.append(file)

for file in os.listdir(image_path_dark_perturber):
    if file.endswith('.npy'):
        im_dp.append(file)

for file in os.listdir(image_path_luminous_perturber):
    if file.endswith('.npy'):
        im_lp.append(file)

In [ ]:
#Setting the norm and color scale
resid_norm =mcolors.TwoSlopeNorm(vmin=-0.025,vcenter=0,vmax=0.025)

#Defining details of the grid
fig,axs = plt.subplots(10,10,figsize=(10,10))
n_cols = 10

for i in range(0,sample_num-1):
    imris = axs[i//n_cols,i%n_cols].imshow(np.load(image_path_luminous_perturber+im_lp[i])-np.load(image_path_main_deflector+im_md[i]), norm=resid_norm,cmap='bwr')
    axs[i//n_cols,i%n_cols].set_xticks([])
    axs[i//n_cols,i%n_cols].set_yticks([])

fig.colorbar(imris, ax=axs)
plt.show()
fig.savefig('../docs/figures/lp_resid_sample.png',bbox_inches='tight')

### 4) Calculate Metrics ###

Now we will calculate the standard deviation, accuracy, and bias metrics for the predicted parameters compared to the truth values.

In [ ]:
#Generating metrics for standard deviation, accuracy, and bias
sample_num = 101
param_num = 10
mean_metrics = metrics.PerturberSampleTrunc(sample_num, y_test, y_pred_wop, y_pred_wp, y_pred_wpl, std_pred_wop, std_pred_wp, std_pred_wpl)

#Saving metric values to a csv file
np.savetxt('../data_tables/metrics_base.csv', mean_metrics, fmt="%1.2f", delimiter=",")

### 5) Interpret Output from the Network ###

Here we are plotting corner plots for ten of the simulated lenses in each sample to see how the two dimension posteriors of the ten lens parameters compare to the known truth values.

In [ ]:
# Defining the learning parameters and their names
learning_params_names = [r'$\theta_\mathrm{E}$',r'$\gamma_1$',r'$\gamma_2$',r'$\gamma_\mathrm{lens}$',r'$e_1$',
								r'$e_2$',r'$x_{lens}$',r'$y_{lens}$',r'$x_{src}$',r'$y_{src}$']

#Generating corner plots for 10 lenses, this range can be user defined.
for i in range(0,10):
    posterior_samples_wop = multivariate_normal(mean=y_pred_wop[i],cov=np.linalg.inv(prec_pred_wop[i])).rvs(size=int(5e3))
    posterior_samples_wp = multivariate_normal(mean=y_pred_wp[i],cov=np.linalg.inv(prec_pred_wp[i])).rvs(size=int(5e3))
    posterior_samples_wpl = multivariate_normal(mean=y_pred_wpl[i],cov=np.linalg.inv(prec_pred_wpl[i])).rvs(size=int(5e3))

    fig = corner.corner(posterior_samples_wop,labels=np.asarray(learning_params_names),bins=20,
                show_titles=False,plot_datapoints=False,label_kwargs=dict(fontsize=30),
                levels=[0.68,0.95],color='slategray',fill_contours=True,smooth=1.0,
                hist_kwargs={'density':True,'color':'slategray','lw':3},title_fmt='.2f',max_n_ticks=3,fig=None,
                truths=y_test[i],
                truth_color='black')
    corner.corner(posterior_samples_wp,labels=np.asarray(learning_params_names),bins=20,
                show_titles=False,plot_datapoints=False,label_kwargs=dict(fontsize=30),
                levels=[0.68,0.95],color='firebrick',fill_contours=True,smooth=1.0,
                hist_kwargs={'density':True,'color':'firebrick','lw':3},title_fmt='.2f',max_n_ticks=3,fig=fig)
    corner.corner(posterior_samples_wpl,labels=np.asarray(learning_params_names),bins=20,
                show_titles=False,plot_datapoints=False,label_kwargs=dict(fontsize=30),
                levels=[0.68,0.95],color='goldenrod',fill_contours=True,smooth=1.0,
                hist_kwargs={'density':True,'color':'goldenrod','lw':3},title_fmt='.2f',max_n_ticks=3,fig=fig)

    color = ['slategray', 'firebrick', 'goldenrod']
    label = ['Without Perturber', 'With Perturber', 'With Perturber Light']
    axes = np.array(fig.axes).reshape(param_num, param_num)
    axes[0,param_num-2].legend(handles=[mlines.Line2D([], [], color=color[i], label=label[i]) for i in range(0,3)],frameon=False,
                fontsize=30,loc=10)

    im_wop = np.load(image_path_main_deflector+im_md[i])
    im_wp = np.load(image_path_dark_perturber+im_dp[i])
    im_wpl = np.load(image_path_luminous_perturber+im_lp[i])
    axes[0,3].imshow(im_wop)
    axes[0,4].imshow(im_wp)
    axes[0,5].imshow(im_wpl)

    plt.savefig('../docs/figures/test_sample_corner_plots/corner_plot_'+str(i)+'.png')